# Lesson 7B: Anomaly Detection Practical — Fraud Detection with Isolation Forest, LOF, One-Class SVM, and an Autoencoder

## Introduction

Lesson 7A derived three anomaly-detection paradigms from scratch on a clean synthetic dataset with a generous 9% contamination rate. Real fraud detection looks nothing like that: fraud is typically **well under 1-2%** of transactions, features interact in non-obvious ways, and you almost never have a labeled validation set to tune a threshold against in production.

This lesson closes that gap:

1. Build a **fraud-style dataset** with realistic severe class imbalance
2. Apply **Isolation Forest, LOF, and One-Class SVM** (scikit-learn implementations — the theory and from-scratch derivations are in 7A) to it
3. Introduce a **fourth paradigm**: a **PyTorch autoencoder**, trained only on normal transactions, that flags anomalies by **reconstruction error**
4. Compare all four with **precision, recall, and PR-AUC** — not ROC-AUC, which is misleadingly optimistic under severe imbalance
5. Discuss **threshold selection without a labeled validation set** — the threshold is fundamentally a business decision (cost of a false alarm vs. cost of a missed fraud), not a purely statistical one

In this lesson:
1. Generate a fraud-style dataset with ~2% positive class
2. Apply the three 7A methods via scikit-learn
3. Train a PyTorch autoencoder on normal-only data and score by reconstruction error
4. Compare precision/recall/PR-AUC across all four methods
5. Select an operating threshold without touching the labels

## Table of Contents

1. [Introduction](#introduction)
2. [Required Libraries](#required-libraries)
3. [Fraud-Style Dataset](#dataset)
4. [Isolation Forest, LOF, One-Class SVM (scikit-learn)](#three-methods)
5. [PyTorch Autoencoder: Architecture and Training](#autoencoder)
6. [Reconstruction Error as an Anomaly Score](#reconstruction-error)
7. [Why PR-AUC, Not ROC-AUC](#pr-auc)
8. [Comparing All Four Methods](#comparison)
9. [Threshold Selection Without Labels](#threshold)
10. [Conclusion](#conclusion)

<a name="required-libraries"></a>
## Required Libraries

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.datasets import make_classification
from sklearn.ensemble import IsolationForest
from sklearn.neighbors import LocalOutlierFactor
from sklearn.svm import OneClassSVM
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import precision_recall_curve, average_precision_score, roc_auc_score, precision_score, recall_score

np.random.seed(42)
torch.manual_seed(42)
plt.style.use('seaborn-v0_8-darkgrid')
%matplotlib inline

print("✅ Libraries loaded!")

<a name="dataset"></a>
## Fraud-Style Dataset

We simulate a transactions dataset with 10 features and a **2% fraud rate** — realistic for card-fraud and similar tabular fraud problems. Labels exist here only so we can *evaluate* every method; none of the four models see the fraud label at training time.

In [ ]:
X, y = make_classification(
    n_samples=8000, n_features=10, n_informative=6, n_redundant=2,
    n_clusters_per_class=2, weights=[0.98, 0.02], flip_y=0.001,
    class_sep=1.2, random_state=42,
)

X = StandardScaler().fit_transform(X)

print(f"Total transactions: {len(X)}")
print(f"Fraud rate: {y.mean():.2%} ({y.sum()} of {len(y)})")

fig, ax = plt.subplots(figsize=(7, 5))
ax.bar(['Normal', 'Fraud'], [np.sum(y == 0), np.sum(y == 1)], color=['steelblue', 'crimson'])
ax.set_title('Class Imbalance', fontsize=12, fontweight='bold')
ax.set_ylabel('Count')
for i, v in enumerate([np.sum(y == 0), np.sum(y == 1)]):
    ax.text(i, v, str(v), ha='center', va='bottom', fontweight='bold')
plt.tight_layout()
plt.show()

<a name="three-methods"></a>
## Isolation Forest, LOF, One-Class SVM (scikit-learn)

All three are fit on the **full unlabeled dataset** (`contamination` set to our best guess at the fraud rate) — the standard unsupervised deployment: no fraud labels available at training time.

In [ ]:
contamination = y.mean()  # in practice this is an estimate, not ground truth

iforest = IsolationForest(n_estimators=200, contamination=contamination, random_state=42)
iforest.fit(X)
scores_iforest = -iforest.score_samples(X)

lof = LocalOutlierFactor(n_neighbors=20, contamination=contamination)
lof.fit_predict(X)
scores_lof = -lof.negative_outlier_factor_

ocsvm = OneClassSVM(kernel='rbf', nu=contamination, gamma='scale')
ocsvm.fit(X)
scores_ocsvm = -ocsvm.decision_function(X)

for name, scores in [('Isolation Forest', scores_iforest), ('LOF', scores_lof), ('One-Class SVM', scores_ocsvm)]:
    print(f"{name:20s}  ROC-AUC={roc_auc_score(y, scores):.3f}  PR-AUC={average_precision_score(y, scores):.3f}")

<a name="autoencoder"></a>
## PyTorch Autoencoder: Architecture and Training

### Core Insight

An autoencoder learns to **compress and reconstruct** its input through a narrow bottleneck. Trained only on normal transactions, it becomes good at reconstructing *normal* patterns — and correspondingly bad at reconstructing fraud, since fraud wasn't part of what it learned to compress. **Reconstruction error becomes the anomaly score.**

This is the practical assumption for training: plenty of historical data is normal-labeled (or safely assumed normal), while confirmed fraud is rare — exactly the imbalance that makes this paradigm attractive.

### Architecture

A small symmetric encoder-decoder: `10 → 8 → 4 → 8 → 10`, with ReLU activations and a linear (unconstrained) output layer, since inputs are standardized (not bounded to [0, 1]).

In [ ]:
class Autoencoder(nn.Module):
    def __init__(self, input_dim: int, bottleneck_dim: int = 4):
        super().__init__()
        self.encoder = nn.Sequential(
            nn.Linear(input_dim, 8),
            nn.ReLU(),
            nn.Linear(8, bottleneck_dim),
            nn.ReLU(),
        )
        self.decoder = nn.Sequential(
            nn.Linear(bottleneck_dim, 8),
            nn.ReLU(),
            nn.Linear(8, input_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.decoder(self.encoder(x))


# Train on normal-only data — the model never sees a fraud example during training
X_train_normal = X[y == 0]
X_train_tensor = torch.FloatTensor(X_train_normal)
X_all_tensor = torch.FloatTensor(X)

autoencoder = Autoencoder(input_dim=X.shape[1], bottleneck_dim=4)
optimizer = optim.Adam(autoencoder.parameters(), lr=1e-3)
loss_fn = nn.MSELoss()

n_epochs = 60
batch_size = 128
losses = []

for epoch in range(n_epochs):
    perm = torch.randperm(len(X_train_tensor))
    epoch_loss = 0.0
    for start in range(0, len(perm), batch_size):
        idx = perm[start:start + batch_size]
        batch = X_train_tensor[idx]

        optimizer.zero_grad()
        reconstruction = autoencoder(batch)
        loss = loss_fn(reconstruction, batch)
        loss.backward()
        optimizer.step()
        epoch_loss += loss.item() * len(idx)

    losses.append(epoch_loss / len(X_train_tensor))

plt.figure(figsize=(7, 4))
plt.plot(losses)
plt.title('Autoencoder Training Loss (normal-only data)', fontsize=12, fontweight='bold')
plt.xlabel('Epoch')
plt.ylabel('Mean squared reconstruction error')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print(f"Final training loss: {losses[-1]:.4f}")

<a name="reconstruction-error"></a>
## Reconstruction Error as an Anomaly Score

Score every transaction (normal and fraud) by its per-sample mean squared reconstruction error. If training worked, fraud should reconstruct noticeably worse than normal transactions, even though the model never saw a single labeled fraud example.

In [ ]:
with torch.no_grad():
    reconstructions = autoencoder(X_all_tensor)
    scores_autoencoder = ((reconstructions - X_all_tensor) ** 2).mean(dim=1).numpy()

print(f"Mean reconstruction error (normal): {scores_autoencoder[y == 0].mean():.4f}")
print(f"Mean reconstruction error (fraud):  {scores_autoencoder[y == 1].mean():.4f}")
print(f"ROC-AUC={roc_auc_score(y, scores_autoencoder):.3f}  PR-AUC={average_precision_score(y, scores_autoencoder):.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
ax.hist(scores_autoencoder[y == 0], bins=60, alpha=0.6, label='Normal', color='steelblue', density=True)
ax.hist(scores_autoencoder[y == 1], bins=60, alpha=0.6, label='Fraud', color='crimson', density=True)
ax.set_title('Reconstruction Error Distribution', fontsize=12, fontweight='bold')
ax.set_xlabel('Mean squared reconstruction error')
ax.set_ylabel('Density')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a name="pr-auc"></a>
## Why PR-AUC, Not ROC-AUC

ROC-AUC compares true-positive rate against **false**-positive rate — and under 2% contamination, the pool of true negatives is so large that even a mediocre model racks up a deceptively high ROC-AUC. **Precision-Recall AUC** instead tracks precision (of the transactions you flag, how many are really fraud?) against recall, which stays sensitive to the imbalance: a naive classifier that flags everything gets ROC-AUC around 0.5 but a PR-AUC that collapses to the base fraud rate (here, ~2%).

Rule of thumb: **under severe class imbalance, PR-AUC is the metric that reflects the real cost of a fraud system — too many false alarms exhausts investigator time, too many misses costs money directly.**

In [ ]:
fig, ax = plt.subplots(figsize=(8, 6))

results = {}
for name, scores in [
    ('Isolation Forest', scores_iforest),
    ('LOF', scores_lof),
    ('One-Class SVM', scores_ocsvm),
    ('Autoencoder', scores_autoencoder),
]:
    precision, recall, _ = precision_recall_curve(y, scores)
    pr_auc = average_precision_score(y, scores)
    roc_auc = roc_auc_score(y, scores)
    results[name] = {'pr_auc': pr_auc, 'roc_auc': roc_auc, 'scores': scores}
    ax.plot(recall, precision, label=f'{name} (PR-AUC={pr_auc:.3f})', linewidth=2)

ax.axhline(y.mean(), color='gray', linestyle='--', label=f'No-skill baseline ({y.mean():.3f})')
ax.set_xlabel('Recall')
ax.set_ylabel('Precision')
ax.set_title('Precision-Recall Curves: All Four Methods', fontsize=12, fontweight='bold')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

<a name="comparison"></a>
## Comparing All Four Methods

In [ ]:
summary = pd.DataFrame([
    {'Method': name, 'ROC-AUC': r['roc_auc'], 'PR-AUC': r['pr_auc']}
    for name, r in results.items()
]).sort_values('PR-AUC', ascending=False).reset_index(drop=True)

print(summary.to_string(index=False))
print(f"\nNo-skill PR-AUC baseline (= fraud rate): {y.mean():.3f}")
print("💡 Note how close ROC-AUC scores are across methods, while PR-AUC spreads them out — PR-AUC is the more discriminating metric here")

<a name="threshold"></a>
## Threshold Selection Without Labels

In production there is usually **no labeled validation set** to pick a threshold against — that's the entire premise of unsupervised anomaly detection. Two practical approaches that don't require labels:

1. **Contamination-rate thresholding**: flag the top-$k$% of scores, where $k$ is a business estimate of the expected anomaly rate (this is what `contamination`/`nu` already encodes for the scikit-learn models).
2. **Elbow detection**: sort scores ascending and look for the point of maximum curvature — where scores start rising sharply. This assumes a reasonably clean population/anomaly separation, which is not guaranteed, but requires no assumption about the anomaly rate.

We apply the elbow method to the autoencoder's reconstruction error, then check (using labels, for illustration only) how the resulting threshold performs.

In [ ]:
def elbow_threshold(scores: np.ndarray) -> float:
    """Point of maximum perpendicular distance from the line joining the sorted curve's endpoints."""
    sorted_scores = np.sort(scores)
    n = len(sorted_scores)
    x = np.arange(n)

    # Normalize both axes to [0, 1] so the "distance from the line" comparison is scale-free
    x_norm = x / (n - 1)
    y_norm = (sorted_scores - sorted_scores.min()) / (sorted_scores.max() - sorted_scores.min())

    # Vector from first to last point defines the reference line
    line_vec = np.array([x_norm[-1] - x_norm[0], y_norm[-1] - y_norm[0]])
    line_vec_norm = line_vec / np.linalg.norm(line_vec)

    points = np.column_stack([x_norm - x_norm[0], y_norm - y_norm[0]])
    proj_length = points @ line_vec_norm
    proj_points = np.outer(proj_length, line_vec_norm)
    distances = np.linalg.norm(points - proj_points, axis=1)

    elbow_idx = np.argmax(distances)
    return sorted_scores[elbow_idx]


threshold = elbow_threshold(scores_autoencoder)
predicted_anomaly = scores_autoencoder >= threshold

flagged_rate = predicted_anomaly.mean()
precision = precision_score(y, predicted_anomaly)
recall = recall_score(y, predicted_anomaly)

print(f"Elbow-selected threshold: {threshold:.4f}")
print(f"Flagged as anomalous: {flagged_rate:.2%} of transactions (true fraud rate: {y.mean():.2%})")
print(f"Precision: {precision:.3f}  Recall: {recall:.3f}")

fig, ax = plt.subplots(figsize=(8, 5))
sorted_scores = np.sort(scores_autoencoder)
ax.plot(sorted_scores, linewidth=1.5)
ax.axhline(threshold, color='red', linestyle='--', label=f'Elbow threshold ({threshold:.4f})')
ax.set_title('Sorted Reconstruction Error with Elbow Threshold', fontsize=12, fontweight='bold')
ax.set_xlabel('Transaction rank (sorted by score)')
ax.set_ylabel('Reconstruction error')
ax.legend()
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("💡 The elbow method needs no assumed contamination rate, but assumes a visible knee in the sorted curve — verify it visually before trusting it")

<a name="conclusion"></a>
## Conclusion

### Key Takeaways

1. **Realistic fraud data is severely imbalanced** — 1-2% positive class is typical, not the 9% used for illustration in 7A.
2. **Isolation Forest, LOF, and One-Class SVM transfer directly** from 7A to this harder setting, applied via scikit-learn with `contamination`/`nu` set to a best-guess fraud rate.
3. **Autoencoders add a fourth, reconstruction-based paradigm** — train only on normal data, score by how badly a point reconstructs.
4. **PR-AUC, not ROC-AUC, is the right comparison metric** under severe imbalance — ROC-AUC compresses differences that matter for a fraud team's actual workload.
5. **Threshold selection is a business decision** — contamination-rate thresholding requires a rate estimate; elbow detection requires none but assumes a visible knee in the score distribution. Neither replaces judgment about the cost of false alarms vs. missed fraud.

### When to Use Which (Practical Guidance)

- **Isolation Forest**: strong default for large, high-dimensional tabular fraud data — fast, robust, few hyperparameters.
- **LOF**: when normal behavior itself has multiple density regimes (e.g., different customer segments with different "normal" spending patterns).
- **One-Class SVM**: when you can afford careful $\nu$/kernel tuning and want an explicit decision boundary; less practical at very large scale.
- **Autoencoder**: when you have ample normal-only data and want a model that can flexibly capture nonlinear structure — and a natural extension point toward the VAE-based approaches in Lesson 12.

### Next Steps

This closes Lesson 7. In Lesson 8, we move from spotting rare points to mining frequent patterns: association rules and the Apriori algorithm for market-basket analysis.

You now have a complete practical toolkit for anomaly detection: three classical unsupervised paradigms plus a deep-learning approach, evaluated the way a real fraud team would — on precision, recall, and the trade-off between them 🎯